In [1]:
import ir_datasets
import ir_datasets_owi

ir_datasets_owi.register()
# dataset = ir_datasets.load("owi/dev")
# dataset = ir_datasets.load("owi/test")
dataset = ir_datasets.load("owi/subsampled/dev")
dataset = ir_datasets.load("owi/subsampled/test")



In [2]:
# Get the first document
doc = next(dataset.docs_iter())

# Print all fields (namedtuple fields)
print(doc._fields)  # if it's a namedtuple

# Or as a dictionary
print(doc._asdict())

('doc_id', 'url', 'main_content', 'title', 'description')
{'doc_id': '00766356da28858a1059367fc5c7eb78dfb216f7eb2ee57292e8bb6e98d8aecc', 'url': 'https://abcnews.go.com/Politics/migrant-surge-continues-test-authorities-south-texas/story?id=80125537', 'main_content': '<h1>Migrant surge continues to test authorities in south Texas</h1>\n\n<p>Video showing officials using horses for crowd control is stirring controversy.</p>\n\nSeptember 20, 2021, 8:34 PM\n<ul>\n  <li></li>\n  <li></li>\n  <li></li>\n  <li></li>\n</ul>\n\n<p>As the Biden administration continues to grapple with a surge of Haitian migrants on the border, images have surfaced showing the tactics used by the U.S. Border Patrol agents riding horses to control the crowd around the river.</p>\n\n<p>Video from news outlets <a href="https://www.washingtonpost.com/politics/2021/09/20/what-one-photo-border-tells-us-about-evolving-migrant-crisis/">Reuters</a> and <a href="https://www.youtube.com/watch?v=UTFnKJqcPks">Al Jazeera</a> ap

In [3]:
for q in dataset.queries_iter():
    print(q)


GenericQuery(query_id='1', text='american civil war')
GenericQuery(query_id='2', text='white shoes cleaning')
GenericQuery(query_id='5', text='Safest vehicles')
GenericQuery(query_id='6', text='Nautical mile')
GenericQuery(query_id='9', text='biggest church milan')
GenericQuery(query_id='10', text='best networking methods')
GenericQuery(query_id='11', text='allergy friendly cats')
GenericQuery(query_id='12', text='Hiphop dance competitions')
GenericQuery(query_id='14', text='LeBron James GOAT debate')
GenericQuery(query_id='17', text='Lung Cancer')
GenericQuery(query_id='19', text='Nuclear energy France')
GenericQuery(query_id='21', text='Gartner hype cycle')
GenericQuery(query_id='22', text='easiest filament 3d printing')
GenericQuery(query_id='25', text='Oasis famous songs')
GenericQuery(query_id='26', text='100 men versus gorilla')
GenericQuery(query_id='27', text='who is the ceo of steam')
GenericQuery(query_id='28', text='maximum particulate matter levels Europe')
GenericQuery(que

In [4]:
print("Number of docs:", dataset.docs_count())
print("Number of queries:", dataset.queries_count())


Number of docs: 357212
Number of queries: None


In [5]:

print(doc._fields)  # if it's a namedtuple

('doc_id', 'url', 'main_content', 'title', 'description')


In [13]:
import json
from pathlib import Path
import ir_datasets

docs= dataset.docs_iter()
# json file creation
# writing documents into file json
output_folder = Path("data/owi_samplejsonl")
output_folder.mkdir(parents=True, exist_ok=True)
output_path = output_folder / "docs.jsonl"

with open(output_path, "w", encoding="utf-8") as f_out:
    for doc in docs:
        if(doc.title != None and doc.main_content != None):
            text = (doc.title + " " + doc.main_content).strip() if hasattr(doc, "title") else doc.text
            record = {"id": doc.doc_id, "text": text}
            f_out.write(json.dumps(record, ensure_ascii=False) + "\n")


In [ ]:
%%bash

export JAVA_HOME=/usr/lib/jvm/java-21-openjdk
export PATH=$JAVA_HOME/bin:$PATH

python -m pyserini.index.lucene \
  --collection JsonCollection \
  --input data/owi_samplejsonl \
  --index pyserini_indexes/owi_sample_lucineindex \
  --generator DefaultLuceneDocumentGenerator \
  --threads 32 \
  --storePositions \
  --storeDocvectors \
  --storeRaw

In [ ]:
%%bash

export JAVA_HOME=/usr/lib/jvm/java-21-openjdk
export PATH=$JAVA_HOME/bin:$PATH

python -m pyserini.encode \
  input \
    --corpus data/owi_samplejsonl/docs.jsonl\
    --fields text \
    --shard-id 0 \
    --shard-num 1 \
  output \
    --embeddings colbert_encoded_docs \
    --to-faiss \
  encoder \
    --encoder castorini/tct_colbert-v2-hnp-msmarco \
    --fields text \
    --batch 16 \
    --fp16

355134it [00:14, 24217.35it/s]
  0%|          | 0/22196 [00:00<?, ?it/s]/home/luuk/Uni/IR/project/myvenv/lib/python3.10/site-packages/pyserini/encode/_tct_colbert.py:67: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
  9%|▉         | 2049/22196 [03:07<22:22, 15.00it/s]  

In [ ]:
from pyserini.search.lucene import LuceneSearcher

# Load your local Lucene index
searcher = LuceneSearcher('/Users/luuk/Uni/IR/2025IRProject/pyserini_indexes/owi_sample_lucineindex')

# Perform a search
hits = searcher.search('how to walk', k=5)

# Retrieve document content
for i, hit in enumerate(hits):
    print(f'Rank {i+1}')
    print(f'DocID: {hit.docid}')
    print(f'Score: {hit.score:.5f}')
    print(f'Text snippet: {hit.raw[:500]}')  # first 500 characters of the document
    print('-' * 80)



JavaException: JVM exception occurred: java.lang.IllegalArgumentException: pyserini_indexes/owi_sample_luceneindex does not exist or is not a directory.
java.lang.IllegalArgumentException: pyserini_indexes/owi_sample_luceneindex does not exist or is not a directory.
	io.anserini.search.SimpleSearcher.<init>(SimpleSearcher.java:130)
	io.anserini.search.SimpleSearcher.<init>(SimpleSearcher.java:115)